In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# --- Imports & Config ---
import os, numpy as np
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T, torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import global_mean_pool, MessagePassing
from torch_geometric.utils import add_self_loops, softmax as pyg_softmax
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from skimage.segmentation import slic
from scipy import ndimage
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

DATASET_ROOT = "/kaggle/input/datasets/raghbirsingh/brain-tumor-ds"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")
# Auto-detect validation dir
for vname in ["validation", "valid", "val"]:
    _vp = os.path.join(DATASET_ROOT, vname)
    if os.path.isdir(_vp):
        VAL_DIR = _vp; break
else:
    VAL_DIR = None

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = 4
IMAGE_SIZE = 224
RESNET_FEATURE_DIM = 2048
GAT_HIDDEN_DIM = 256
NUM_HEADS = 4
DROPOUT = 0.3
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 30
OUTPUT_DIR = "output"
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "enhanced_model.pth")
os.makedirs(PLOTS_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
MEAN_NP = np.array(IMAGENET_MEAN)
STD_NP = np.array(IMAGENET_STD)
print(f"Device: {device} | Val dir: {VAL_DIR}")

In [ ]:
# --- Dataset ---
def get_transform(train=False):
    if train:
        return T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.3),T.RandomRotation(15),
            T.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.1),
            T.ToTensor(),T.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    return T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),T.ToTensor(),
        T.Normalize(IMAGENET_MEAN,IMAGENET_STD)])

class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform or get_transform()
        self.class_to_idx = {n:i for i,n in enumerate(CLASS_NAMES)}
        self.image_paths, self.labels = [], []
        for cd in sorted(Path(root_dir).iterdir()):
            if not cd.is_dir(): continue
            name = cd.name.lower()
            if name not in self.class_to_idx: continue
            lbl = self.class_to_idx[name]
            for f in sorted(cd.glob("*")):
                if f.suffix.lower() in (".png",".jpg",".jpeg"):
                    self.image_paths.append(str(f)); self.labels.append(lbl)
        print(f"  Loaded {len(self.image_paths)} from {root_dir}")
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]


In [ ]:
# --- CBAM ---
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch//r, 8)
        self.mlp = nn.Sequential(nn.Linear(ch,mid,bias=False),nn.ReLU(True),nn.Linear(mid,ch,bias=False))
    def forward(self, x):
        B,C,_,_ = x.shape
        a = self.mlp(x.mean([2,3])); m = self.mlp(x.amax([2,3]))
        return x * torch.sigmoid(a+m).view(B,C,1,1)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2,1,k,padding=k//2,bias=False)
    def forward(self, x):
        d = torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], 1)
        return x * torch.sigmoid(self.conv(d))

class CBAM(nn.Module):
    def __init__(self, ch, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(ch, r); self.sa = SpatialAttention(k)
    def forward(self, x): return self.sa(self.ca(x))


In [ ]:
# --- ResNet50 + CBAM Backbone (Part 3) ---
class ResNet50CBAMBackbone(nn.Module):
    """ResNet50 with CBAM after each major block. [B,3,224,224]->[B,2048,7,7]"""
    def __init__(self, pretrained=True, freeze=False):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(resnet.conv1,resnet.bn1,resnet.relu,resnet.maxpool)
        self.layer1=resnet.layer1; self.cbam1=CBAM(256)
        self.layer2=resnet.layer2; self.cbam2=CBAM(512)
        self.layer3=resnet.layer3; self.cbam3=CBAM(1024)
        self.layer4=resnet.layer4; self.cbam4=CBAM(2048)
        if freeze:
            for p in self.parameters(): p.requires_grad=False
    def forward(self, x):
        x=self.stem(x)
        x=self.cbam1(self.layer1(x))
        x=self.cbam2(self.layer2(x))
        x=self.cbam3(self.layer3(x))
        x=self.cbam4(self.layer4(x))
        return x

In [ ]:
# --- Superpixel Graph Construction ---
def _unnorm(t):
    img=t.detach().cpu().numpy().transpose(1,2,0)
    return np.clip(img*STD_NP+MEAN_NP,0,1)

def build_superpixel_graph(img_t, feat_map, label, n_seg=50, compact=10):
    C,H,W = feat_map.shape
    img_np = _unnorm(img_t)
    segs = slic(img_np, n_segments=n_seg, compactness=compact, start_label=0, channel_axis=2)
    sh,sw = segs.shape[0]/H, segs.shape[1]/W
    sd = ndimage.zoom(segs.astype(float),(1/sh,1/sw),order=0).astype(int)[:H,:W]
    feat = feat_map.detach().cpu()
    uids = np.unique(sd); id2i = {s:i for i,s in enumerate(uids)}; N=len(uids)
    nf = torch.zeros(N, C)
    for s in uids:
        mask = torch.from_numpy((sd==s).astype(np.float32))
        cnt = mask.sum().clamp(min=1)
        nf[id2i[s]] = (feat*mask.unsqueeze(0)).sum([1,2])/cnt
    src,dst=[],[]
    for r in range(H):
        for c in range(W):
            cur=sd[r,c]
            for dr,dc in [(0,1),(1,0),(1,1),(1,-1)]:
                nr,nc=r+dr,c+dc
                if 0<=nr<H and 0<=nc<W:
                    nb=sd[nr,nc]
                    if cur!=nb:
                        i,j=id2i[cur],id2i[nb]; src+=[i,j]; dst+=[j,i]
    if not src:
        for i in range(N):
            for j in range(i+1,N): src+=[i,j]; dst+=[j,i]
    es=set(zip(src,dst))
    if es: s,d=zip(*es)
    else: s,d=[],[]
    ei=torch.tensor([list(s),list(d)],dtype=torch.long)
    return Data(x=nf,edge_index=ei,y=torch.tensor([label],dtype=torch.long))

def build_multiscale_graphs(img_t, feat_map, label):
    fine=build_superpixel_graph(img_t,feat_map,label,n_seg=50,compact=10)
    coarse=build_superpixel_graph(img_t,feat_map,label,n_seg=15,compact=30)
    return fine, coarse


In [ ]:
# --- Class-Aware GAT ---
class ClassAwareGATConv(MessagePassing):
    def __init__(self, in_ch, out_ch, n_cls=4, heads=4, drop=0.3, concat=True, ced=32):
        super().__init__(aggr='add', node_dim=0)
        self.out_channels=out_ch; self.heads=heads; self.concat=concat; self.dropout=drop
        self.lin=nn.Linear(in_ch, heads*out_ch, bias=False)
        self.class_embed=nn.Embedding(n_cls, ced)
        self.class_proj=nn.Linear(ced, heads, bias=False)
        self.att_src=nn.Parameter(torch.empty(1,heads,out_ch))
        self.att_dst=nn.Parameter(torch.empty(1,heads,out_ch))
        self.bias=nn.Parameter(torch.zeros(heads*out_ch if concat else out_ch))
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)
    def forward(self, x, edge_index, class_logits=None):
        H,C=self.heads,self.out_channels
        xp=self.lin(x).view(-1,H,C)
        as_=( xp*self.att_src).sum(-1); ad=(xp*self.att_dst).sum(-1)
        if class_logits is not None and class_logits.dim()==2:
            pr=F.softmax(class_logits.detach(),-1)
            ce=pr@self.class_embed.weight
            self._cb=self.class_proj(ce).mean(0)
        else: self._cb=torch.zeros(H,device=x.device)
        ei,_=add_self_loops(edge_index,num_nodes=x.size(0))
        out=self.propagate(ei,x=xp,alpha=(as_,ad))
        if self.concat: return out.view(-1,H*C)+self.bias
        return out.mean(1)+self.bias
    def message(self, x_j, alpha_j, alpha_i, index, ptr, size_i):
        a=alpha_i+alpha_j+self._cb.unsqueeze(0)
        a=F.leaky_relu(a,0.2); a=pyg_softmax(a,index,ptr,size_i)
        a=F.dropout(a,p=self.dropout,training=self.training)
        return x_j*a.unsqueeze(-1)


In [ ]:
# --- Dual GAT + Fusion ---
class DualGATClassifier(nn.Module):
    def __init__(self, ind=2048, hid=256, nc=4, h=4, d=0.3):
        super().__init__(); self.drop=d
        self.fg1=ClassAwareGATConv(ind,hid,nc,h,d,True); self.fb1=nn.BatchNorm1d(hid*h)
        self.fg2=ClassAwareGATConv(hid*h,hid,nc,1,d,False); self.fb2=nn.BatchNorm1d(hid)
        self.cg1=ClassAwareGATConv(ind,hid,nc,h,d,True); self.cb1=nn.BatchNorm1d(hid*h)
        self.cg2=ClassAwareGATConv(hid*h,hid,nc,1,d,False); self.cb2=nn.BatchNorm1d(hid)
        self.gate=nn.Sequential(nn.Linear(hid*2,hid),nn.Sigmoid())
        self.fc1=nn.Linear(hid,64); self.fc2=nn.Linear(64,32); self.fc3=nn.Linear(32,nc)
        self.bn1=nn.BatchNorm1d(64); self.bn2=nn.BatchNorm1d(32)
    def _pipe(self,x,ei,b,g1,b1,g2,b2,cl=None):
        x=F.elu(b1(g1(x,ei,cl))); x=F.dropout(x,self.drop,self.training)
        x=F.elu(b2(g2(x,ei,cl))); return global_mean_pool(x,b)
    def forward(self,fx,fei,fb,cx,cei,cb,cl=None):
        f=self._pipe(fx,fei,fb,self.fg1,self.fb1,self.fg2,self.fb2,cl)
        c=self._pipe(cx,cei,cb,self.cg1,self.cb1,self.cg2,self.cb2,cl)
        cat=torch.cat([f,c],-1); g=self.gate(cat); fused=g*f+(1-g)*c
        x=F.dropout(F.elu(self.bn1(self.fc1(fused))),self.drop,self.training)
        x=F.dropout(F.elu(self.bn2(self.fc2(x))),self.drop,self.training)
        return F.log_softmax(self.fc3(x),-1)


In [ ]:
# --- Full Model ---
class EnhancedResNet50GAT(nn.Module):
    def __init__(self, pretrained=True, freeze=False):
        super().__init__()
        self.backbone = ResNet50CBAMBackbone(pretrained, freeze)
        self.gat = DualGATClassifier()
    def forward(self, images, labels=None):
        dev=images.device; B=images.size(0)
        feat_maps=self.backbone(images)
        fine_list, coarse_list = [], []
        for i in range(B):
            lbl = labels[i].item() if labels is not None else 0
            fg, cg = build_multiscale_graphs(images[i], feat_maps[i], lbl)
            fine_list.append(fg); coarse_list.append(cg)
        fb=Batch.from_data_list(fine_list).to(dev)
        cb=Batch.from_data_list(coarse_list).to(dev)
        return self.gat(fb.x,fb.edge_index,fb.batch,cb.x,cb.edge_index,cb.batch)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [ ]:
# --- Training & Eval Functions ---
def train_one_epoch(model, loader, optimizer):
    model.train(); tl,cor,tot=0.0,0,0
    for imgs,lbls in loader:
        imgs,lbls=imgs.to(device),lbls.to(device)
        optimizer.zero_grad(); out=model(imgs,lbls)
        loss=F.nll_loss(out,lbls); loss.backward(); optimizer.step()
        tl+=loss.item()*lbls.size(0); cor+=(out.argmax(1)==lbls).sum().item(); tot+=lbls.size(0)
    return tl/tot, cor/tot

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); tl,cor,tot=0.0,0,0; yt,yp=[],[]
    for imgs,lbls in loader:
        imgs,lbls=imgs.to(device),lbls.to(device)
        out=model(imgs,lbls); loss=F.nll_loss(out,lbls); p=out.argmax(1)
        tl+=loss.item()*lbls.size(0); cor+=(p==lbls).sum().item(); tot+=lbls.size(0)
        yt.extend(lbls.cpu().numpy()); yp.extend(p.cpu().numpy())
    return tl/tot, cor/tot, np.array(yt), np.array(yp)

def print_metrics(yt,yp,name="Test"):
    a=accuracy_score(yt,yp); p=precision_score(yt,yp,average="weighted",zero_division=0)
    r=recall_score(yt,yp,average="weighted",zero_division=0)
    f=f1_score(yt,yp,average="weighted",zero_division=0)
    print(f"\n{'='*50}\n  {name} Metrics\n{'='*50}")
    print(f"  Acc:{a:.6f} Prec:{p:.6f} Rec:{r:.6f} F1:{f:.6f}")
    print(classification_report(yt,yp,target_names=CLASS_NAMES,zero_division=0))
    return {"accuracy":a,"precision":p,"recall":r,"f1":f}

def plot_confusion_matrix(yt,yp,name="Test"):
    cm=confusion_matrix(yt,yp); plt.figure(figsize=(8,6))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{name} Confusion Matrix")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/cm_{name.lower()}.png",dpi=150); plt.show()

def plot_curves(h):
    ep=range(1,len(h["tl"])+1); fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
    a1.plot(ep,h["tl"],label="Train"); a1.plot(ep,h["vl"],label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.legend(); a1.grid(alpha=0.3)
    a2.plot(ep,h["ta"],label="Train"); a2.plot(ep,h["va"],label="Val")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Accuracy"); a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/curves.png",dpi=150); plt.show()


In [ ]:
# --- Load Data ---
print("\n"+"="*60+"\n  Loading Datasets\n"+"="*60)
train_ds = BrainTumorDataset(TRAIN_DIR, get_transform(True))
test_ds = BrainTumorDataset(TEST_DIR, get_transform(False))
if VAL_DIR:
    val_ds = BrainTumorDataset(VAL_DIR, get_transform(False))
else:
    from torch.utils.data import random_split
    vs=int(0.15*len(train_ds)); ts=len(train_ds)-vs
    train_ds, val_ds = random_split(train_ds,[ts,vs])
    print(f"  No val dir found, split train: {ts} val: {vs}")

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)


In [ ]:
# --- Build & Train ---
print("\n"+"="*60+"\n  Building Enhanced Model\n"+"="*60)
model = EnhancedResNet50GAT(pretrained=True, freeze=False).to(device)
print(f"  Trainable params: {model.count_parameters():,}")

optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,"min",factor=0.5,patience=5,min_lr=1e-7)
hist={"tl":[],"vl":[],"ta":[],"va":[]}; best=0.0

print(f"\n  Training {NUM_EPOCHS} epochs...")
for ep in range(1,NUM_EPOCHS+1):
    tl,ta=train_one_epoch(model,train_loader,optimizer)
    vl,va,_,_=evaluate(model,val_loader); scheduler.step(vl)
    hist["tl"].append(tl);hist["vl"].append(vl);hist["ta"].append(ta);hist["va"].append(va)
    lr=optimizer.param_groups[0]["lr"]
    print(f"Ep {ep:3d}/{NUM_EPOCHS} | TrL {tl:.4f} TrA {ta:.4f} | VL {vl:.4f} VA {va:.4f} | LR {lr:.2e}")
    if va>best: best=va; torch.save(model.state_dict(),MODEL_SAVE_PATH); print(f"  * Best {va:.4f}")

print(f"\nTraining done. Best val acc: {best:.4f}")


In [ ]:
# --- Plots & Eval ---
plot_curves(hist)
model.load_state_dict(torch.load(MODEL_SAVE_PATH,map_location=device))
train_eval=BrainTumorDataset(TRAIN_DIR,get_transform(False))
train_eval_loader=DataLoader(train_eval,batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
for nm,ld in [("Train",train_eval_loader),("Val",val_loader),("Test",test_loader)]:
    _,_,yt,yp=evaluate(model,ld); print_metrics(yt,yp,nm); plot_confusion_matrix(yt,yp,nm)


In [ ]:
#--- Grad-CAM Visualization ---
class GradCAM:
    def __init__(self, model):
        self.model=model; self.act=None; self.grad=None
        model.backbone.layer4.register_forward_hook(lambda m,i,o: setattr(self,'act',o.detach()))
        model.backbone.layer4.register_full_backward_hook(lambda m,gi,go: setattr(self,'grad',go[0].detach()))
    def generate(self, img_t, tc=None):
        self.model.eval(); img_t.requires_grad_(True)
        out=self.model(img_t)
        if tc is None: tc=out.argmax(1).item()
        self.model.zero_grad(); out[0,tc].backward()
        w=self.grad.mean([2,3],keepdim=True)
        cam=F.relu((w*self.act).sum(1,keepdim=True))
        cam=F.interpolate(cam,(224,224),mode='bilinear',align_corners=False)
        cam=cam.squeeze().cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam, tc

def visualize_gradcam(model, image_path, save_path=None):
    tfm=get_transform(False)
    img=Image.open(image_path).convert("RGB")
    it=tfm(img).unsqueeze(0).to(device)
    gcam=GradCAM(model); hm,pc=gcam.generate(it)
    inp=np.array(img.resize((224,224)))/255.0
    overlay=0.5*inp+0.5*plt.cm.jet(hm)[:,:,:3]
    fig,ax=plt.subplots(1,3,figsize=(15,5))
    ax[0].imshow(inp);ax[0].set_title("Original")
    ax[1].imshow(hm,cmap='jet');ax[1].set_title("Grad-CAM")
    ax[2].imshow(overlay);ax[2].set_title(f"Overlay - Pred: {CLASS_NAMES[pc]}")
    for a in ax: a.axis('off')
    plt.tight_layout()
    if save_path: plt.savefig(save_path,dpi=150)
    plt.show(); return hm,pc

# GAT Attention Visualization
def visualize_gat_attention(model, image_tensor, save_path=None):
    model.eval()
    with torch.no_grad():
        if image_tensor.dim()==3: image_tensor=image_tensor.unsqueeze(0)
        image_tensor=image_tensor.to(device)
        fm=model.backbone(image_tensor)
        fg,_=build_multiscale_graphs(image_tensor[0].cpu(),fm[0].cpu(),0)
        fg=fg.to(device)
        g1=model.gat.fg1; H=g1.heads; C=g1.out_channels
        xp=g1.lin(fg.x).view(-1,H,C)
        as_=(xp*g1.att_src).sum(-1); ad=(xp*g1.att_dst).sum(-1)
        ni=torch.zeros(fg.x.size(0),device=device)
        ei=fg.edge_index
        for e in range(ei.size(1)):
            s,d=ei[0,e].item(),ei[1,e].item()
            ni[d]+=abs((as_[s]+ad[d]).mean().item())
        ni=ni.cpu().numpy(); ni=(ni-ni.min())/(ni.max()-ni.min()+1e-8)
        fig,ax=plt.subplots(figsize=(8,6))
        sc=ax.scatter(range(len(ni)),ni,c=ni,cmap='hot',s=100,edgecolors='black')
        ax.set_xlabel("Node (Superpixel)"); ax.set_ylabel("Attention")
        ax.set_title("GAT Node Attention (Fine Scale)")
        plt.colorbar(sc,label="Importance"); plt.tight_layout()
        if save_path: plt.savefig(save_path,dpi=150)
        plt.show(); return ni



In [ ]:
# --- Visualization ---
print("\n"+"="*60+"\n  Explainability Visualizations\n"+"="*60)
# Grad-CAM on first test image
sample_path = test_ds.image_paths[0]
print(f"  Grad-CAM on: {sample_path}")
visualize_gradcam(model, sample_path, f"{PLOTS_DIR}/gradcam.png")

# GAT attention on first test image
sample_img, _ = test_ds[0]
visualize_gat_attention(model, sample_img, f"{PLOTS_DIR}/gat_attention.png")

print("\n Pipeline complete!")
print(f"   Model: {MODEL_SAVE_PATH}")


In [ ]:
# %% [markdown]
# # Enhanced Hybrid ResNet50-CBAM + Multi-Scale Class-Aware GAT
# Classes: Glioma | Meningioma | No Tumor | Pituitary
# Features: Superpixel graphs, CBAM, Class-Aware GAT, Dual-scale fusion, Grad-CAM

# %% --- Cell 1: Install ---
# !pip install torch torchvision torch-geometric scikit-learn seaborn matplotlib Pillow scikit-image opencv-python-headless -q

# %% --- Cell 2: Imports & Config ---
import os, numpy as np
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T, torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import global_mean_pool, MessagePassing
from torch_geometric.utils import add_self_loops, softmax as pyg_softmax
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from skimage.segmentation import slic
from scipy import ndimage
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

DATASET_ROOT = "/kaggle/input/datasets/raghbirsingh/brain-tumor-ds"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")
# Auto-detect validation dir
for vname in ["validation", "valid", "val"]:
    _vp = os.path.join(DATASET_ROOT, vname)
    if os.path.isdir(_vp):
        VAL_DIR = _vp; break
else:
    VAL_DIR = None

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = 4
IMAGE_SIZE = 224
RESNET_FEATURE_DIM = 2048
GAT_HIDDEN_DIM = 256
NUM_HEADS = 4
DROPOUT = 0.3
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 2
OUTPUT_DIR = "output"
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "enhanced_model.pth")
os.makedirs(PLOTS_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
MEAN_NP = np.array(IMAGENET_MEAN)
STD_NP = np.array(IMAGENET_STD)
print(f"Device: {device} | Val dir: {VAL_DIR}")

# %% --- Cell 3: Dataset ---
def get_transform(train=False):
    if train:
        return T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.3),T.RandomRotation(15),
            T.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.1),
            T.ToTensor(),T.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    return T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),T.ToTensor(),
        T.Normalize(IMAGENET_MEAN,IMAGENET_STD)])

class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform or get_transform()
        self.class_to_idx = {n:i for i,n in enumerate(CLASS_NAMES)}
        self.image_paths, self.labels = [], []
        for cd in sorted(Path(root_dir).iterdir()):
            if not cd.is_dir(): continue
            name = cd.name.lower()
            if name not in self.class_to_idx: continue
            lbl = self.class_to_idx[name]
            for f in sorted(cd.glob("*")):
                if f.suffix.lower() in (".png",".jpg",".jpeg"):
                    self.image_paths.append(str(f)); self.labels.append(lbl)
        print(f"  Loaded {len(self.image_paths)} from {root_dir}")
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]

# %% --- Cell 4: CBAM (Part 3) ---
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch//r, 8)
        self.mlp = nn.Sequential(nn.Linear(ch,mid,bias=False),nn.ReLU(True),nn.Linear(mid,ch,bias=False))
    def forward(self, x):
        B,C,_,_ = x.shape
        a = self.mlp(x.mean([2,3])); m = self.mlp(x.amax([2,3]))
        return x * torch.sigmoid(a+m).view(B,C,1,1)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2,1,k,padding=k//2,bias=False)
    def forward(self, x):
        d = torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], 1)
        return x * torch.sigmoid(self.conv(d))

class CBAM(nn.Module):
    def __init__(self, ch, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(ch, r); self.sa = SpatialAttention(k)
    def forward(self, x): return self.sa(self.ca(x))

# %% --- Cell 5: ResNet50 + CBAM Backbone (Part 3) ---
class ResNet50CBAMBackbone(nn.Module):
    """ResNet50 with CBAM after each major block. [B,3,224,224]->[B,2048,7,7]"""
    def __init__(self, pretrained=True, freeze=False):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(resnet.conv1,resnet.bn1,resnet.relu,resnet.maxpool)
        self.layer1=resnet.layer1; self.cbam1=CBAM(256)
        self.layer2=resnet.layer2; self.cbam2=CBAM(512)
        self.layer3=resnet.layer3; self.cbam3=CBAM(1024)
        self.layer4=resnet.layer4; self.cbam4=CBAM(2048)
        if freeze:
            for p in self.parameters(): p.requires_grad=False
    def forward(self, x):
        x=self.stem(x)
        x=self.cbam1(self.layer1(x))
        x=self.cbam2(self.layer2(x))
        x=self.cbam3(self.layer3(x))
        x=self.cbam4(self.layer4(x))
        return x

# %% --- Cell 6: Superpixel Graph Construction (Part 1) ---
def _unnorm(t):
    img=t.cpu().numpy().transpose(1,2,0)
    return np.clip(img*STD_NP+MEAN_NP,0,1)

def build_superpixel_graph(img_t, feat_map, label, n_seg=50, compact=10):
    C,H,W = feat_map.shape
    img_np = _unnorm(img_t)
    segs = slic(img_np, n_segments=n_seg, compactness=compact, start_label=0, channel_axis=2)
    sh,sw = segs.shape[0]/H, segs.shape[1]/W
    sd = ndimage.zoom(segs.astype(float),(1/sh,1/sw),order=0).astype(int)[:H,:W]
    feat = feat_map.detach().cpu()
    uids = np.unique(sd); id2i = {s:i for i,s in enumerate(uids)}; N=len(uids)
    nf = torch.zeros(N, C)
    for s in uids:
        mask = torch.from_numpy((sd==s).astype(np.float32))
        cnt = mask.sum().clamp(min=1)
        nf[id2i[s]] = (feat*mask.unsqueeze(0)).sum([1,2])/cnt
    src,dst=[],[]
    for r in range(H):
        for c in range(W):
            cur=sd[r,c]
            for dr,dc in [(0,1),(1,0),(1,1),(1,-1)]:
                nr,nc=r+dr,c+dc
                if 0<=nr<H and 0<=nc<W:
                    nb=sd[nr,nc]
                    if cur!=nb:
                        i,j=id2i[cur],id2i[nb]; src+=[i,j]; dst+=[j,i]
    if not src:
        for i in range(N):
            for j in range(i+1,N): src+=[i,j]; dst+=[j,i]
    es=set(zip(src,dst))
    if es: s,d=zip(*es)
    else: s,d=[],[]
    ei=torch.tensor([list(s),list(d)],dtype=torch.long)
    return Data(x=nf,edge_index=ei,y=torch.tensor([label],dtype=torch.long))

def build_multiscale_graphs(img_t, feat_map, label):
    fine=build_superpixel_graph(img_t,feat_map,label,n_seg=50,compact=10)
    coarse=build_superpixel_graph(img_t,feat_map,label,n_seg=15,compact=30)
    return fine, coarse

# %% --- Cell 7: Class-Aware GAT (Part 4) ---
class ClassAwareGATConv(MessagePassing):
    def __init__(self, in_ch, out_ch, n_cls=4, heads=4, drop=0.3, concat=True, ced=32):
        super().__init__(aggr='add', node_dim=0)
        self.out_channels=out_ch; self.heads=heads; self.concat=concat; self.dropout=drop
        self.lin=nn.Linear(in_ch, heads*out_ch, bias=False)
        self.class_embed=nn.Embedding(n_cls, ced)
        self.class_proj=nn.Linear(ced, heads, bias=False)
        self.att_src=nn.Parameter(torch.empty(1,heads,out_ch))
        self.att_dst=nn.Parameter(torch.empty(1,heads,out_ch))
        self.bias=nn.Parameter(torch.zeros(heads*out_ch if concat else out_ch))
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)
    def forward(self, x, edge_index, class_logits=None):
        H,C=self.heads,self.out_channels
        xp=self.lin(x).view(-1,H,C)
        as_=( xp*self.att_src).sum(-1); ad=(xp*self.att_dst).sum(-1)
        if class_logits is not None and class_logits.dim()==2:
            pr=F.softmax(class_logits.detach(),-1)
            ce=pr@self.class_embed.weight
            self._cb=self.class_proj(ce).mean(0)
        else: self._cb=torch.zeros(H,device=x.device)
        ei,_=add_self_loops(edge_index,num_nodes=x.size(0))
        out=self.propagate(ei,x=xp,alpha=(as_,ad))
        if self.concat: return out.view(-1,H*C)+self.bias
        return out.mean(1)+self.bias
    def message(self, x_j, alpha_j, alpha_i, index, ptr, size_i):
        a=alpha_i+alpha_j+self._cb.unsqueeze(0)
        a=F.leaky_relu(a,0.2); a=pyg_softmax(a,index,ptr,size_i)
        a=F.dropout(a,p=self.dropout,training=self.training)
        return x_j*a.unsqueeze(-1)

# %% --- Cell 8: Dual GAT + Fusion (Part 2) ---
class DualGATClassifier(nn.Module):
    def __init__(self, ind=2048, hid=256, nc=4, h=4, d=0.3):
        super().__init__(); self.drop=d
        self.fg1=ClassAwareGATConv(ind,hid,nc,h,d,True); self.fb1=nn.BatchNorm1d(hid*h)
        self.fg2=ClassAwareGATConv(hid*h,hid,nc,1,d,False); self.fb2=nn.BatchNorm1d(hid)
        self.cg1=ClassAwareGATConv(ind,hid,nc,h,d,True); self.cb1=nn.BatchNorm1d(hid*h)
        self.cg2=ClassAwareGATConv(hid*h,hid,nc,1,d,False); self.cb2=nn.BatchNorm1d(hid)
        self.gate=nn.Sequential(nn.Linear(hid*2,hid),nn.Sigmoid())
        self.fc1=nn.Linear(hid,64); self.fc2=nn.Linear(64,32); self.fc3=nn.Linear(32,nc)
        self.bn1=nn.BatchNorm1d(64); self.bn2=nn.BatchNorm1d(32)
    def _pipe(self,x,ei,b,g1,b1,g2,b2,cl=None):
        x=F.elu(b1(g1(x,ei,cl))); x=F.dropout(x,self.drop,self.training)
        x=F.elu(b2(g2(x,ei,cl))); return global_mean_pool(x,b)
    def forward(self,fx,fei,fb,cx,cei,cb,cl=None):
        f=self._pipe(fx,fei,fb,self.fg1,self.fb1,self.fg2,self.fb2,cl)
        c=self._pipe(cx,cei,cb,self.cg1,self.cb1,self.cg2,self.cb2,cl)
        cat=torch.cat([f,c],-1); g=self.gate(cat); fused=g*f+(1-g)*c
        x=F.dropout(F.elu(self.bn1(self.fc1(fused))),self.drop,self.training)
        x=F.dropout(F.elu(self.bn2(self.fc2(x))),self.drop,self.training)
        return F.log_softmax(self.fc3(x),-1)

# %% --- Cell 9: Full Model ---
class EnhancedResNet50GAT(nn.Module):
    def __init__(self, pretrained=True, freeze=False):
        super().__init__()
        self.backbone = ResNet50CBAMBackbone(pretrained, freeze)
        self.gat = DualGATClassifier()
    def forward(self, images, labels=None):
        dev=images.device; B=images.size(0)
        feat_maps=self.backbone(images)
        fine_list, coarse_list = [], []
        for i in range(B):
            lbl = labels[i].item() if labels is not None else 0
            fg, cg = build_multiscale_graphs(images[i], feat_maps[i], lbl)
            fine_list.append(fg); coarse_list.append(cg)
        fb=Batch.from_data_list(fine_list).to(dev)
        cb=Batch.from_data_list(coarse_list).to(dev)
        return self.gat(fb.x,fb.edge_index,fb.batch,cb.x,cb.edge_index,cb.batch)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# %% --- Cell 10: Training & Eval Functions ---
def train_one_epoch(model, loader, optimizer):
    model.train(); tl,cor,tot=0.0,0,0
    for imgs,lbls in loader:
        imgs,lbls=imgs.to(device),lbls.to(device)
        optimizer.zero_grad(); out=model(imgs,lbls)
        loss=F.nll_loss(out,lbls); loss.backward(); optimizer.step()
        tl+=loss.item()*lbls.size(0); cor+=(out.argmax(1)==lbls).sum().item(); tot+=lbls.size(0)
    return tl/tot, cor/tot

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); tl,cor,tot=0.0,0,0; yt,yp=[],[]
    for imgs,lbls in loader:
        imgs,lbls=imgs.to(device),lbls.to(device)
        out=model(imgs,lbls); loss=F.nll_loss(out,lbls); p=out.argmax(1)
        tl+=loss.item()*lbls.size(0); cor+=(p==lbls).sum().item(); tot+=lbls.size(0)
        yt.extend(lbls.cpu().numpy()); yp.extend(p.cpu().numpy())
    return tl/tot, cor/tot, np.array(yt), np.array(yp)

def print_metrics(yt,yp,name="Test"):
    a=accuracy_score(yt,yp); p=precision_score(yt,yp,average="weighted",zero_division=0)
    r=recall_score(yt,yp,average="weighted",zero_division=0)
    f=f1_score(yt,yp,average="weighted",zero_division=0)
    print(f"\n{'='*50}\n  {name} Metrics\n{'='*50}")
    print(f"  Acc:{a:.6f} Prec:{p:.6f} Rec:{r:.6f} F1:{f:.6f}")
    print(classification_report(yt,yp,target_names=CLASS_NAMES,zero_division=0))
    return {"accuracy":a,"precision":p,"recall":r,"f1":f}

def plot_confusion_matrix(yt,yp,name="Test"):
    cm=confusion_matrix(yt,yp); plt.figure(figsize=(8,6))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{name} Confusion Matrix")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/cm_{name.lower()}.png",dpi=150); plt.show()

def plot_curves(h):
    ep=range(1,len(h["tl"])+1); fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
    a1.plot(ep,h["tl"],label="Train"); a1.plot(ep,h["vl"],label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.legend(); a1.grid(alpha=0.3)
    a2.plot(ep,h["ta"],label="Train"); a2.plot(ep,h["va"],label="Val")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Accuracy"); a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/curves.png",dpi=150); plt.show()

# %% --- Cell 11: Load Data ---
print("\n"+"="*60+"\n  Loading Datasets\n"+"="*60)
train_ds = BrainTumorDataset(TRAIN_DIR, get_transform(True))
test_ds = BrainTumorDataset(TEST_DIR, get_transform(False))
if VAL_DIR:
    val_ds = BrainTumorDataset(VAL_DIR, get_transform(False))
else:
    from torch.utils.data import random_split
    vs=int(0.15*len(train_ds)); ts=len(train_ds)-vs
    train_ds, val_ds = random_split(train_ds,[ts,vs])
    print(f"  No val dir found, split train: {ts} val: {vs}")

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)

# %% --- Cell 12: Build & Train ---
print("\n"+"="*60+"\n  Building Enhanced Model\n"+"="*60)
model = EnhancedResNet50GAT(pretrained=True, freeze=False).to(device)
print(f"  Trainable params: {model.count_parameters():,}")

optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,"min",factor=0.5,patience=5,min_lr=1e-7)
hist={"tl":[],"vl":[],"ta":[],"va":[]}; best=0.0

print(f"\n  Training {NUM_EPOCHS} epochs...")
for ep in range(1,NUM_EPOCHS+1):
    tl,ta=train_one_epoch(model,train_loader,optimizer)
    vl,va,_,_=evaluate(model,val_loader); scheduler.step(vl)
    hist["tl"].append(tl);hist["vl"].append(vl);hist["ta"].append(ta);hist["va"].append(va)
    lr=optimizer.param_groups[0]["lr"]
    print(f"Ep {ep:3d}/{NUM_EPOCHS} | TrL {tl:.4f} TrA {ta:.4f} | VL {vl:.4f} VA {va:.4f} | LR {lr:.2e}")
    if va>best: best=va; torch.save(model.state_dict(),MODEL_SAVE_PATH); print(f"  * Best {va:.4f}")

print(f"\nTraining done. Best val acc: {best:.4f}")

# %% --- Cell 13: Plots & Eval ---
plot_curves(hist)
model.load_state_dict(torch.load(MODEL_SAVE_PATH,map_location=device))
train_eval=BrainTumorDataset(TRAIN_DIR,get_transform(False))
train_eval_loader=DataLoader(train_eval,batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
for nm,ld in [("Train",train_eval_loader),("Val",val_loader),("Test",test_loader)]:
    _,_,yt,yp=evaluate(model,ld); print_metrics(yt,yp,nm); plot_confusion_matrix(yt,yp,nm)

# %% --- Cell 14: Grad-CAM Visualization (Part 5) ---
class GradCAM:
    def __init__(self, model):
        self.model=model; self.act=None; self.grad=None
        model.backbone.layer4.register_forward_hook(lambda m,i,o: setattr(self,'act',o.detach()))
        model.backbone.layer4.register_full_backward_hook(lambda m,gi,go: setattr(self,'grad',go[0].detach()))
    def generate(self, img_t, tc=None):
        self.model.eval(); img_t.requires_grad_(True)
        out=self.model(img_t)
        if tc is None: tc=out.argmax(1).item()
        self.model.zero_grad(); out[0,tc].backward()
        w=self.grad.mean([2,3],keepdim=True)
        cam=F.relu((w*self.act).sum(1,keepdim=True))
        cam=F.interpolate(cam,(224,224),mode='bilinear',align_corners=False)
        cam=cam.squeeze().cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam, tc

def visualize_gradcam(model, image_path, save_path=None):
    tfm=get_transform(False)
    img=Image.open(image_path).convert("RGB")
    it=tfm(img).unsqueeze(0).to(device)
    gcam=GradCAM(model); hm,pc=gcam.generate(it)
    inp=np.array(img.resize((224,224)))/255.0
    overlay=0.5*inp+0.5*plt.cm.jet(hm)[:,:,:3]
    fig,ax=plt.subplots(1,3,figsize=(15,5))
    ax[0].imshow(inp);ax[0].set_title("Original")
    ax[1].imshow(hm,cmap='jet');ax[1].set_title("Grad-CAM")
    ax[2].imshow(overlay);ax[2].set_title(f"Overlay - Pred: {CLASS_NAMES[pc]}")
    for a in ax: a.axis('off')
    plt.tight_layout()
    if save_path: plt.savefig(save_path,dpi=150)
    plt.show(); return hm,pc

# GAT Attention Visualization
def visualize_gat_attention(model, image_tensor, save_path=None):
    model.eval()
    with torch.no_grad():
        if image_tensor.dim()==3: image_tensor=image_tensor.unsqueeze(0)
        image_tensor=image_tensor.to(device)
        fm=model.backbone(image_tensor)
        fg,_=build_multiscale_graphs(image_tensor[0].cpu(),fm[0].cpu(),0)
        fg=fg.to(device)
        g1=model.gat.fg1; H=g1.heads; C=g1.out_channels
        xp=g1.lin(fg.x).view(-1,H,C)
        as_=(xp*g1.att_src).sum(-1); ad=(xp*g1.att_dst).sum(-1)
        ni=torch.zeros(fg.x.size(0),device=device)
        ei=fg.edge_index
        for e in range(ei.size(1)):
            s,d=ei[0,e].item(),ei[1,e].item()
            ni[d]+=abs((as_[s]+ad[d]).mean().item())
        ni=ni.cpu().numpy(); ni=(ni-ni.min())/(ni.max()-ni.min()+1e-8)
        fig,ax=plt.subplots(figsize=(8,6))
        sc=ax.scatter(range(len(ni)),ni,c=ni,cmap='hot',s=100,edgecolors='black')
        ax.set_xlabel("Node (Superpixel)"); ax.set_ylabel("Attention")
        ax.set_title("GAT Node Attention (Fine Scale)")
        plt.colorbar(sc,label="Importance"); plt.tight_layout()
        if save_path: plt.savefig(save_path,dpi=150)
        plt.show(); return ni

# %% --- Cell 15: Example Visualization Usage ---
print("\n"+"="*60+"\n  Explainability Visualizations\n"+"="*60)
# Grad-CAM on first test image
sample_path = test_ds.image_paths[0]
print(f"  Grad-CAM on: {sample_path}")
visualize_gradcam(model, sample_path, f"{PLOTS_DIR}/gradcam_example.png")

# GAT attention on first test image
sample_img, _ = test_ds[0]
visualize_gat_attention(model, sample_img, f"{PLOTS_DIR}/gat_attention.png")

print("\n Pipeline complete!")
print(f"   Model: {MODEL_SAVE_PATH}")


Device: cuda | Val dir: /kaggle/input/datasets/raghbirsingh/brain-tumor-ds/valid

  Loading Datasets
  Loaded 5600 from /kaggle/input/datasets/raghbirsingh/brain-tumor-ds/train
  Loaded 1200 from /kaggle/input/datasets/raghbirsingh/brain-tumor-ds/test
  Loaded 1200 from /kaggle/input/datasets/raghbirsingh/brain-tumor-ds/valid

  Building Enhanced Model
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 181MB/s] 


  Trainable params: 29,087,148

  Training 2 epochs...
Ep   1/2 | TrL 0.7661 TrA 0.7400 | VL 0.4800 VA 0.8767 | LR 1.00e-04
  * Best 0.8767
Ep   2/2 | TrL 0.5783 TrA 0.8107 | VL 0.4236 VA 0.8867 | LR 1.00e-04
  * Best 0.8867

Training done. Best val acc: 0.8867
  Loaded 5600 from /kaggle/input/datasets/raghbirsingh/brain-tumor-ds/train
